In [1]:
from PySide2.QtCore import QTimer, QTime, Signal
import time
import numpy as np
import os
import os
import sys

# Get the absolute path of the current script
script_path = os.path.abspath(r"C:\Users\YY3\GIT\squdi\src\qudi\jupyternotebooks\hom")
# Get the directory name of the script path
script_dir = os.path.dirname(script_path)
# Change the working directory to the script's directory
os.chdir(script_dir)
from hom import auto
from hom.auto import *
from hom.tools import *
%gui qt

In [2]:
import importlib

importlib.reload(auto)

<module 'hom.auto' from 'C:\\Users\\YY3\\GIT\\squdi\\src\\qudi\\jupyternotebooks\\hom\\auto.py'>

In [3]:
folder_save = r'Z:\Vlad\SnV\TPI\Electrodes_e4\F2\atto3-def1\bf_def1\24-07-attempt-I'
current_cryo = 'atto3' #fix the notation of current and non-active cryo
non_active_cryo = 'bf'
integrate_for_mins = 180
values = [0, -.2, -.5, -0.8, -1.13]  #the counts have to be equalized!

params = {
    'ple_gui': ple_gui,
    'laser_scanner_logic': laser_scanner_logic,
    'scanner_gui' : scanner_gui,
    'scanning_data_logic' : scanning_data_logic,
    'pulsestreamer' : pulsestreamer,
    'timetaggerlogic': timetaggerlogic,
    'timetagger': timetagger,
    'timetagger_remote': timetagger_remote,
    'poi_manager_logic_remote': poi_manager_logic_remote,
    'switchlogic': switchlogic,
    'ibeam_smart_remote': ibeam_smart_remote,
    'powercontroller_logic': powercontroller_logic,
    'integrate_for_mins': integrate_for_mins,
    'current_cryo': current_cryo,
    'non_active_cryo': non_active_cryo,
    'folder_save': folder_save,
    'values': values,
}

measurement_e_hom = auto.StarkHOM(ao_electrodes_remote, **params)  

#adjust bf green min and max positions
measurement_e_hom.min_position = 110
measurement_e_hom.max_position = 190
measurement_e_hom.max_power = 50e3

In [5]:
measurement_e_hom.measurement_mode('Off-res')

In [7]:
measurement_e_hom.equilize_powers()

(11.301, 11.434000000000001, 10.223)

In [8]:
measurement_e_hom.measurement_mode('PLE')

In [4]:
ao_electrodes_remote.setpoint=-1.0

In [12]:
timetagger_remote._mw.count_display_label.text()[-4:]

'kc/s'

In [5]:
measurement_e_hom.align_resonances(offset=0, dv = 0.2, steps = 25)

(array([972.2 , 666.6 , 716.6 ,   1.81,   5.52,   4.81,   1.7 , 644.4 ,
         6.13,   1.11, 788.8 , 777.7 ,   4.53,   2.06,   2.49,   2.74,
         3.48,   3.81,   5.45,   5.73,   6.14,   7.06,   8.62,   8.7 ,
        10.03]), array([-1.1       , -1.09166667, -1.08333333, -1.075     , -1.06666667,
       -1.05833333, -1.05      , -1.04166667, -1.03333333, -1.025     ,
       -1.01666667, -1.00833333, -1.        , -0.99166667, -0.98333333,
       -0.975     , -0.96666667, -0.95833333, -0.95      , -0.94166667,
       -0.93333333, -0.925     , -0.91666667, -0.90833333, -0.9       ]))

In [71]:
v0 = ao_electrodes_remote.setpoint
dv = 0.2
steps = 30
counts = np.array([])
for _setpoint in (setpoints := np.linspace(v0 - dv/2, v0 + dv/2, steps)):
    ao_electrodes_remote.setpoint = _setpoint
    
    cts = float(timetagger_remote._mw.count_display_label.text()[:5])
    counts = np.append(counts, cts)
    time.sleep(0.1)
# ao_electrodes_remote.setpoint = v0
ao_electrodes_remote.setpoint = setpoints[np.argmax(counts)]

In [70]:
ao_electrodes_remote.setpoint = setpoints[np.argmax(counts)]

In [69]:
counts

array([7.97, 1.53, 1.61, 1.71, 2.12, 2.16, 2.88, 3.11, 4.23, 5.1 , 5.52,
       7.42, 8.85, 9.54, 9.81, 8.82, 7.47, 6.06, 5.32, 4.06])

In [19]:
measurement_e_hom.align_resonances(offset=0)

In [ ]:

measurement_e_hom.start_periodic_refocus(refocus_period_mins = 30, 
                               count_check_period_sec = 60, 
                               resonance_refocus_mis = 5,
                               bf_refocus_mins = 360)